# 04 - PHASE 1 GATE (Pl@ntNet): reliability and predictability of delta_y

AGENTS.md Sec 6 - the experiment that decides the project. **Nothing under `pcc/method/` may be
built until this passes.**

- **Gate A (6.2)** split-half reliability of delta_y >= 0.30. The CEILING on any achievable R2.
- **Gate B (6.3)** held-out-CLASS R2 clearly > 0, normalized by the ceilings.
- **Gate C (6.3)** geometry must beat log-prevalence-only and distance-only.
- **Sec 6.4** the predicted correction must buy efficiency on held-out classes.

## Amendment 5 governs how this is read - please read before the numbers

Descriptor stability on Pl@ntNet never reached 0.90 (best 0.815 at q=50) and the tail quartile is
**flat at ~0.68** because those classes hold only 2-7 images - no quota can fix it. **Sec 3.3 is
therefore NOT satisfied**, and three consequences follow, all human-approved:

1. **Gate B is read asymmetrically.** Descriptor noise only ATTENUATES R2; it cannot manufacture
   predictability. A **PASS is meaningful**; a **FAIL is AMBIGUOUS**, not evidence of no signal.
2. **Gate C is computed WITHIN prevalence strata (PRIMARY)**, pooled only as a secondary view.
   Descriptor accuracy tracks prevalence (head-tail spread **+0.238**), so the pooled comparison is
   confounded. Validated on planted stories: prevalence-driven delta -> geometry beats prevalence in
   **0/4** strata; geometry-driven delta -> **4/4**. Pooled R2 was 0.935 vs 0.928, i.e. useless for
   telling them apart.
3. **PRIMARY feature set = the 4 features with stability >= 0.90**; `full` is the sensitivity view.

## Amendment 2: delta_y at matched n_cal, plus a prevalence null

The quantile estimator's bias depends on group size and `n_y` tracks prevalence, so unmatched
delta_y correlates with prevalence even with zero class structure. `n_cal=25` is PRIMARY (152
classes), `n_cal=10` the sensitivity (283 classes); both are pre-registered and both reported.

## Sec 6.4: Amendment 4 design, plus a tail-facing view

Efficiency is measured in the **held-out label space** with the threshold vector free to deflate, so
a constant delta_hat is a genuine no-op. Separately, `pcc.eval.tail` reports macro-coverage per
prevalence stratum - the only way to say anything about the tail, where delta_y is unmeasurable but
coverage aggregated over hundreds of classes is not.

**Scores are OURS** (checkpoint gate did not pass - reports/phase0_checkpoint_gate.md).


## 1. Config - `# === EDIT ME ===`


In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'

DATASET    = 'plantnet'
BACKBONE   = 'resnet50_ltc'
CAL_SPLIT  = 'cal'            # delta_y / calibration
EVAL_SPLIT = 'test'           # evaluation
DESC_SPLIT = 'train_quota'    # descriptors: TRAINING data only (Sec 6.3)

ALPHA      = 0.1              # alpha=0.05 needs n>=19/class (187 classes), 0.01 needs 99 (57)
N_CAL_PRIMARY     = 25        # Amendment 2, pre-registered pair
N_CAL_SENSITIVITY = 10
N_SPLITS_A  = 100             # split-half repetitions (gate A)
N_SPLITS_BC = 100             # class-level splits (gate B/C)
N_NULL      = 30              # prevalence-null repetitions
N_STRATA    = 4
STABLE_THRESHOLD = 0.90       # descriptor-stability cut for the PRIMARY feature set
SEED = 42
EMB_ROOT = f'{DRIVE_ROOT}/embeddings/{DATASET}/{BACKBONE}'
# =======================================================================
print('alpha', ALPHA, '| n_cal primary/sensitivity', N_CAL_PRIMARY, N_CAL_SENSITIVITY)


## 2. Mount Drive + repo + env


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
from pcc.utils.seed import set_seed; from pcc.utils.io import environment_stamp
set_seed(SEED)
print('env:', environment_stamp()['packages'])


## 3. Load: cal scores, eval scores, train embeddings. LEAK GUARD asserted.


In [ ]:
import numpy as np
from pcc.data.load import load_split, load_scores, split_provenance, per_class_counts
from pcc.data.ltc_datasets import NUM_CLASSES
from pcc.scores.base import thr_lac

K = NUM_CLASSES[DATASET]
prov = split_provenance(EMB_ROOT, CAL_SPLIT)
print('scores_source:', prov.get('scores_source'),
      '| under_gate_exception:', prov.get('under_gate_exception'))

sm_cal, y_cal = load_scores(EMB_ROOT, CAL_SPLIT)
sm_ev,  y_ev  = load_scores(EMB_ROOT, EVAL_SPLIT)
S_cal, S_ev = thr_lac(sm_cal), thr_lac(sm_ev)
s_true_cal = S_cal[np.arange(len(y_cal)), y_cal]

dtr = load_split(EMB_ROOT, DESC_SPLIT)
tr_emb = np.asarray(dtr['embeddings']); tr_lab = np.asarray(dtr['labels']).astype(int)
tr_lg  = np.asarray(dtr['logits'])

cal_counts = per_class_counts(y_cal, K)
tr_counts  = per_class_counts(tr_lab, K)
print(f'cal {len(y_cal)} | eval {len(y_ev)} | descriptor images {len(tr_lab)}')
print(f'cal samples/class  median {int(np.median(cal_counts[cal_counts>0]))}')
print(f'train images/class median {int(np.median(tr_counts[tr_counts>0]))}')

# LEAK GUARD (Sec 8.3): descriptors must come from TRAIN, never the calibration split.
# cal/eval come from the val/test directories; descriptors from the train directory.
assert DESC_SPLIT == 'train_quota', 'descriptors must come from the train split'
assert CAL_SPLIT != DESC_SPLIT and EVAL_SPLIT != DESC_SPLIT
print('leak guard OK: descriptors from TRAIN; cal/eval from val/test (disjoint sources)')


## 4. GATE A - split-half reliability of delta_y. Sets the R2 ceiling.


In [ ]:
from pcc.targets.delta import split_half_reliability
from pcc.eval.stats import mean_ci

relA = split_half_reliability(s_true_cal, y_cal, K, ALPHA,
                              n_splits=N_SPLITS_A, seed=SEED)
sp = relA['reliability_splits']; sp = sp[~np.isnan(sp)]
rel_ci = mean_ci(sp)
r_delta = float(relA['reliability_mean'])
print(f"gate A reliability = {r_delta:.3f}  95% CI "
      f"[{rel_ci['ci_low']:.3f}, {rel_ci['ci_high']:.3f}]   (threshold 0.30)")
gate_A_pass = bool(np.isfinite(rel_ci['ci_low']) and rel_ci['ci_low'] >= 0.30)
print(f"  eligible classes (>=4 cal samples): {relA['n_classes_eligible']}")
print(f"  splits producing a value: {relA['n_splits_with_a_value']}/{N_SPLITS_A}")
print(f"  classes contributing per split (mean): {relA['n_classes_contributing_mean']:.0f}")
if relA.get('undefined_reason'):
    print('  UNDEFINED:', relA['undefined_reason'])
print('gate A:', 'PASS' if gate_A_pass else ('UNDEFINED' if not np.isfinite(r_delta) else 'FAIL'))
print(f'(cal median is {int(np.median(cal_counts[cal_counts>0]))} samples/class, so gate A is'
      f' noise-limited; the level-matched estimator is used so small classes still contribute)')


## 5. delta_y at matched n_cal (Amendment 2) + prevalence null


In [ ]:
from pcc.targets.delta import delta_y, delta_y_matched_n, prevalence_null

deltas, nulls = {}, {}
for tag, ncal in (('primary', N_CAL_PRIMARY), ('sensitivity', N_CAL_SENSITIVITY)):
    dm, kept = delta_y_matched_n(s_true_cal, y_cal, K, ALPHA, n_cal=ncal, seed=SEED)
    deltas[tag] = (np.where(kept, dm, np.nan), kept, ncal)
    nl = prevalence_null(s_true_cal, y_cal, K, ALPHA, n_cal=ncal,
                         n_reps=N_NULL, seed=SEED)
    nulls[tag] = nl
    ok = int((kept & np.isfinite(dm)).sum())
    corr = np.nan
    m = kept & np.isfinite(dm) & (cal_counts > 0)
    if m.sum() > 3:
        corr = float(np.corrcoef(dm[m], np.log(cal_counts[m]))[0,1])
    print(f'[{tag}] n_cal={ncal}: delta_y defined for {ok}/{K} classes | '
          f'corr(delta, log n_y)={corr:+.3f}')
    if nl['n_reps'] == 0:
        print('   prevalence null UNDEFINED:', nl.get('undefined_reason'))
    else:
        print(f"   null: mean {nl['null_mean']:+.3f} sd {nl['null_sd']:.3f} "
              f"|null|p95 {nl['null_abs_p95']:.3f} -> observed corr is "
              f"{'WITHIN' if abs(corr) <= nl['null_abs_p95'] else 'BEYOND'} the null")
d_unm = delta_y(s_true_cal, y_cal, K, ALPHA)
print(f'\nunmatched (all samples) delta_y defined for {int(np.isfinite(d_unm).sum())}/{K}'
      f' - reported as sensitivity only (Amendment 2)')


## 6. Descriptors from TRAIN + the pre-registered feature sets


In [ ]:
from pcc.descriptors.phi import build_descriptors
from pcc.descriptors.stability import descriptor_stability, QUOTA_DETERMINED

Phi, names = build_descriptors(tr_emb, tr_lg, tr_lab, K,
                               log_prevalence_from=tr_counts)
print('Phi', Phi.shape, '| finite rows:', int(np.isfinite(Phi).all(axis=1).sum()))

stab = descriptor_stability(tr_emb, tr_lg, tr_lab, K, quotas=(50,), n_reps=3,
                            seed=SEED, stable_threshold=STABLE_THRESHOLD,
                            method='bootstrap')
pf = stab['by_quota'][50]['per_feature']
r_phi = float(stab['by_quota'][50]['mean_corr'])
stable_names = [nm for nm in names if nm not in QUOTA_DETERMINED
                and np.isfinite(pf.get(nm, np.nan)) and pf[nm] >= STABLE_THRESHOLD]
FEATURE_SETS = {'stable': stable_names, 'full': list(names)}
print(f'r_phi (descriptor ceiling, bootstrap q=50) = {r_phi:.3f}')
print('PRIMARY stable set:', stable_names)
print('dropped as unstable:', [nm for nm in names
      if nm not in stable_names and nm not in QUOTA_DETERMINED])
print(f'\nTWO CEILINGS: target r_delta={r_delta:.3f}, descriptor r_phi={r_phi:.3f},'
      f' joint ~{r_delta*r_phi:.3f}. A perfect model cannot exceed the joint bound.')


## 7. GATE B/C - PRIMARY is stratified by prevalence (Amendment 5)


In [ ]:
from pcc.eval.predictability import predictability, predictability_by_stratum

gate_bc = {}
for dtag in ('primary', 'sensitivity'):
    dvec = deltas[dtag][0]
    for fset, feats in FEATURE_SETS.items():
        cols = [names.index(f) for f in feats]
        key = f'{dtag}|{fset}'
        pooled = predictability(Phi[:, cols], dvec, feats, reliability=r_delta,
                                n_splits=N_SPLITS_BC, seed=SEED)
        strat = predictability_by_stratum(Phi[:, cols], dvec, feats, cal_counts,
                                          reliability=r_delta, n_splits=N_SPLITS_BC,
                                          seed=SEED, n_strata=N_STRATA)
        gate_bc[key] = {'pooled': pooled, 'stratified': strat}
        r2 = pooled['r2_by_predictor']['full']
        print(f'=== {key} ===')
        print(f"  POOLED (secondary): R2={r2['mean']:+.3f} "
              f"CI [{r2['ci_low']:+.3f},{r2['ci_high']:+.3f}] gate_B={pooled['gate_B_pass']}")
        print(f"  STRATIFIED (PRIMARY): {strat['n_strata_gate_B_pass']}"
              f"/{strat['n_strata_reported']} strata pass gate B; "
              f"geometry beats: {strat['n_strata_full_beats_ablation']}")
        for sname, sv in strat['summary'].items():
            print(f"    {sname:18s} n={sv['n_classes']:4d} R2={sv['r2_full']:+.3f} "
                  f"B={sv['gate_B_pass']} beats={sv['beats']}")


## 8. Sec 6.4 - efficiency on held-out classes (Amendment 4) + shuffled null


In [ ]:
from pcc.eval.predictability import ridge_fit, ridge_predict
from pcc.eval.setsize import setsize_translation_heldout_space
from pcc.eval.decomposition import group_quantile

dvec = deltas['primary'][0]
cols = [names.index(f) for f in FEATURE_SETS['stable']]
usable = np.where(np.isfinite(dvec) & np.isfinite(Phi).all(axis=1))[0]
print(f'classes usable for Sec 6.4: {len(usable)}')
rng = np.random.default_rng(SEED)
acc = {('class_conditional','obs'): [], ('class_conditional','null'): [],
       ('macro','obs'): [], ('macro','null'): []}
for rep in range(20):
    perm = rng.permutation(usable)
    fit_c, held_c = perm[:len(perm)//2], perm[len(perm)//2:]
    if len(held_c) < 5: continue
    model = ridge_fit(Phi[fit_c][:, cols], dvec[fit_c], 1.0)
    dhat = np.zeros(K); dhat[held_c] = ridge_predict(model, Phi[held_c][:, cols])
    dnull = np.zeros(K); dnull[held_c] = rng.permutation(dhat[held_c])
    qg = group_quantile(s_true_cal, ALPHA, 'empirical')
    for obj in ('class_conditional','macro'):
        for tag, dd in (('obs', dhat), ('null', dnull)):
            try:
                r = setsize_translation_heldout_space(S_ev, y_ev, ALPHA, qg, dd,
                                                     held_c, objective=obj)
                acc[(obj,tag)].append(r['gap'])
            except ValueError:
                pass
sec64 = {}
for obj in ('class_conditional','macro'):
    o = mean_ci(acc[(obj,'obs')]); nl = mean_ci(acc[(obj,'null')])
    beats = bool(o['ci_low'] > nl['ci_high'])
    sec64[obj] = {'observed': o, 'shuffled_null': nl, 'beats_null': beats,
                  'pass': bool(o['ci_low'] > 0 and beats)}
    lbl = 'PRIMARY' if obj=='class_conditional' else 'secondary'
    print(f"[{lbl}] {obj}: gap={o['mean']:+.4f} CI [{o['ci_low']:+.4f},{o['ci_high']:+.4f}] | "
          f"null={nl['mean']:+.4f} | beats_null={beats} -> "
          f"{'PASS' if sec64[obj]['pass'] else 'NOT POSITIVE'}")


## 9. TAIL VIEW - macro-coverage per prevalence stratum

delta_y is unmeasurable for a 2-sample class, but macro-coverage aggregated over hundreds of tail
classes is estimable. This is the only view that says anything about where the claim is supposed to
pay off.


In [ ]:
from pcc.eval.tail import compare_by_stratum

qg = group_quantile(s_true_cal, ALPHA, 'conformal')   # deployment-valid threshold
tail_res = compare_by_stratum(S_ev, y_ev, K, cal_counts, qg, dhat,
                              n_strata=N_STRATA, min_count=1, seed=SEED)
print('stratum'.ljust(22) + 'macro_cov unc -> cor'.rjust(24) + 'size unc -> cor'.rjust(22))
for k in tail_res['uncorrected']:
    if k.startswith('_'): continue
    u, c = tail_res['uncorrected'][k], tail_res['corrected'][k]
    print(k.ljust(22) + f"{u['macro_coverage']:.3f} -> {c['macro_coverage']:.3f}".rjust(24)
          + f"{u['avg_set_size']:.2f} -> {c['avg_set_size']:.2f}".rjust(22))
print('\nunevaluable:', tail_res['uncorrected']['_unevaluable'])


## 10. Sec 9 metric bundle (all metrics together, never size alone)


In [ ]:
from pcc.eval.metrics import summary as sec9_summary
from pcc.eval.setsize import corrected_thresholds
from pcc.eval.conformal import build_sets

heldset = set(int(c) for c in held_c)
group_of_class = {y: ('held_out' if y in heldset else 'seen') for y in range(K)}
sec9 = {}
for arm, thr in (('uncorrected', qg), ('corrected', corrected_thresholds(qg, dhat))):
    sets = build_sets(S_ev, thr)
    sec9[arm] = sec9_summary(sets, y_ev, K, ALPHA, group_of_class=group_of_class)
    print(arm + ':')
    for k, v in sec9[arm].items(): print(f'    {k:24s} {v}')


## 11. Verdict + report, with every mandated caveat attached


In [ ]:
import time
from pcc.utils.io import write_report

def clean(o):
    if isinstance(o, dict): return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): return float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, np.ndarray): return None
    return o

prim = gate_bc['primary|stable']['stratified']
gate_B_primary = prim['n_strata_gate_B_pass'] >= max(1, prim['n_strata_reported'] - 1)
abl = prim['n_strata_full_beats_ablation']
gate_C_primary = all(v >= max(1, prim['n_strata_reported'] - 1) for v in abl.values()) if abl else None

print('GATE A:', 'PASS' if gate_A_pass else 'FAIL')
print('GATE B (stratified, primary):', 'PASS' if gate_B_primary else 'FAIL -> AMBIGUOUS per Sec 3.3')
print('GATE C (stratified, primary):', gate_C_primary, '| per-ablation strata:', abl)
print('Sec 6.4 (class_conditional):', 'PASS' if sec64['class_conditional']['pass'] else 'NOT POSITIVE')

verdict = ('PASS' if (gate_A_pass and gate_B_primary and gate_C_primary
                      and sec64['class_conditional']['pass']) else 'NOT PASSED')
caveats = [
    'Sec 3.3 NOT satisfied: descriptor stability peaked at 0.815 (<0.90), tail quartile flat at',
    '  ~0.68 and unfixable by quota -> a gate-B FAILURE IS AMBIGUOUS, not evidence of no signal.',
    'Descriptor quality tracks prevalence (head-tail spread +0.238) -> gate C is judged on the',
    '  STRATIFIED form; the pooled form cannot separate a geometry story from a prevalence story.',
    'Scores are OURS, not LTC released (checkpoint gate FAILED under a written exception).',
    f'alpha={ALPHA} only; alpha=0.05/0.01 are infeasible for most classes (187/57 of 1081).',
    'Gate A/B/C are computed on an n_cal-restricted, hence PREVALENCE-SELECTED, class subset.',
]
print()
for c in caveats: print('CAVEAT:', c)

report = write_report('pcc/reports', f'04_phase1_gate_{DATASET}',
    hypothesis='delta_y is a reliable class-level signal (A), predictable from class geometry (B) '
               'beyond trivial predictors (C), and the predicted correction buys efficiency on '
               'held-out classes (6.4)',
    pass_criteria='A: split-half reliability CI low >= 0.30. B: held-out R2 CI excludes 0 in '
                  'essentially every prevalence stratum (STRATIFIED is primary per Amendment 5); '
                  'a FAIL is recorded as AMBIGUOUS because Sec 3.3 was not met. C: the full '
                  'descriptor beats log-prevalence-only and distance-only within strata. 6.4: '
                  'held-out-label-space gap > 0 and beating a shuffled null. delta_y at matched '
                  'n_cal (25 primary, 10 sensitivity); both feature sets and both pooled and '
                  'stratified views always reported.',
    config=dict(dataset=DATASET, backbone=BACKBONE, alpha=ALPHA,
                n_cal_primary=N_CAL_PRIMARY, n_cal_sensitivity=N_CAL_SENSITIVITY,
                n_splits_A=N_SPLITS_A, n_splits_BC=N_SPLITS_BC, n_strata=N_STRATA,
                feature_sets={k: list(v) for k, v in FEATURE_SETS.items()},
                stable_threshold=STABLE_THRESHOLD,
                scores_source=prov.get('scores_source'),
                under_gate_exception=bool(prov.get('under_gate_exception')),
                amendments=['#amendment-2','#amendment-4','#amendment-5']),
    seed=SEED,
    results={'gate_A': {'reliability': r_delta, 'ci_low': rel_ci['ci_low'],
                        'ci_high': rel_ci['ci_high'], 'pass': gate_A_pass,
                        'n_classes_eligible': relA['n_classes_eligible'],
                        'n_splits_with_a_value': relA['n_splits_with_a_value'],
                        'n_classes_contributing_mean': relA['n_classes_contributing_mean'],
                        'undefined_reason': relA.get('undefined_reason')},
             'ceilings': {'r_delta': r_delta, 'r_phi': r_phi, 'joint': r_delta*r_phi},
             'delta_y': {t: {'n_cal': deltas[t][2],
                             'n_classes': int((deltas[t][1] & np.isfinite(deltas[t][0])).sum())}
                         for t in deltas},
             'prevalence_null': clean(nulls),
             'gate_BC': clean(gate_bc), 'sec_6_4': clean(sec64),
             'tail_by_stratum': clean(tail_res), 'sec_9_metrics': clean(sec9),
             'gate_B_primary_pass': bool(gate_B_primary),
             'gate_C_primary_pass': gate_C_primary,
             'caveats': caveats},
    conclusion=f'{verdict} - see caveats; a gate-B failure here is AMBIGUOUS (Sec 3.3 unmet)',
    started_at=time.time())
print()
print('report:', report)
print('VERDICT:', verdict)
